# File I/O

Almost every real program reads from or writes to the filesystem — config files, logs, data exports, images, and more. Python gives you a rich set of tools for this, from the simple built-in `open()` all the way to specialised libraries for CSV, JSON, binary data, and temporary files.

## What You'll Learn
1. Opening & closing files — `open()` and the `with` statement
2. Reading files — `read()`, `readline()`, `readlines()`, iteration
3. Writing files — `write()`, `writelines()`, append mode
4. File modes & encoding
5. `pathlib` — modern path handling
6. Working with CSV files
7. Working with JSON files
8. Binary files
9. Temporary files & directories
10. `os` & `shutil` — file system operations
11. File watching & `stat`
12. Large file strategies — streaming & memory-mapping
13. Error handling in file I/O
14. `configparser` — INI config files
15. `pickle` — Python object serialisation

---

In [1]:
# Setup: create a working directory for all notebook output
from pathlib import Path
import os

WORK = Path('file_io_demo')
WORK.mkdir(exist_ok=True)
print(f"Working directory: {WORK.resolve()}")

Working directory: /Users/jacob/Code/Notes/Tutorials/Python-Tutorial/Lessons/01-Beginner/06-FileIO/01-Basics/file_io_demo


---

## 1. Opening & Closing Files

`open()` is the built-in that gives you a file object. Always close files when you're done — the `with` statement does this automatically, even if an exception is raised.

In [2]:
# ❌ Manual open/close — risky if an exception occurs before close()
f = open(WORK / 'hello.txt', 'w')
f.write('Hello, World!\n')
f.close()   # if an exception happens before here, the file leaks

# ✅ The with statement — always preferred
# File is closed automatically when the block exits, even on error
with open(WORK / 'hello.txt', 'w') as f:
    f.write('Hello, World!\n')

print(f"File closed: {f.closed}")

File closed: True


In [3]:
# open() signature:
# open(file, mode='r', buffering=-1, encoding=None, errors=None,
#      newline=None, closefd=True, opener=None)

# The file object has useful attributes
with open(WORK / 'hello.txt', 'r') as f:
    print(f"Name:     {f.name}")
    print(f"Mode:     {f.mode}")
    print(f"Encoding: {f.encoding}")
    print(f"Closed:   {f.closed}")
    content = f.read()

print(f"After with — closed: {f.closed}")
print(f"Content: {content!r}")

Name:     file_io_demo/hello.txt
Mode:     r
Encoding: UTF-8
Closed:   False
After with — closed: True
Content: 'Hello, World!\n'


In [4]:
# Open multiple files at once in a single with statement
with open(WORK / 'source.txt', 'w') as src, \
     open(WORK / 'dest.txt',   'w') as dst:
    src.write('Source content\n')
    dst.write('Destination content\n')

print("Both files written and closed.")

Both files written and closed.


---

## 2. File Modes & Encoding

The `mode` string controls how the file is opened. Getting this right matters.

In [5]:
modes = [
    ('r',   'Read (text)         — default. File must exist.'),
    ('w',   'Write (text)        — creates or TRUNCATES file.'),
    ('a',   'Append (text)       — creates or appends to end.'),
    ('x',   'Exclusive create    — fails if file already exists.'),
    ('r+',  'Read + write        — file must exist, no truncation.'),
    ('w+',  'Write + read        — creates or truncates file.'),
    ('a+',  'Append + read       — creates or appends.'),
    ('rb',  'Read binary         — returns bytes, not str.'),
    ('wb',  'Write binary        — writes bytes.'),
    ('ab',  'Append binary       — appends bytes.'),
    ('rb+', 'Read + write binary — file must exist.'),
]

print(f"{'Mode':<6}  Description")
print('-' * 55)
for mode, desc in modes:
    print(f"  {mode:<4}   {desc}")

Mode    Description
-------------------------------------------------------
  r      Read (text)         — default. File must exist.
  w      Write (text)        — creates or TRUNCATES file.
  a      Append (text)       — creates or appends to end.
  x      Exclusive create    — fails if file already exists.
  r+     Read + write        — file must exist, no truncation.
  w+     Write + read        — creates or truncates file.
  a+     Append + read       — creates or appends.
  rb     Read binary         — returns bytes, not str.
  wb     Write binary        — writes bytes.
  ab     Append binary       — appends bytes.
  rb+    Read + write binary — file must exist.


In [6]:
# Encoding — always specify it for text files!
# If you don't, Python uses the platform default (which varies by OS)

path = WORK / 'unicode_demo.txt'

# Write with explicit UTF-8 encoding
with open(path, 'w', encoding='utf-8') as f:
    f.write('English: Hello\n')
    f.write('Japanese: こんにちは\n')
    f.write('Arabic: مرحبا\n')
    f.write('Emoji: 🐍🎉\n')

# Always read with the same encoding you wrote with
with open(path, 'r', encoding='utf-8') as f:
    print(f.read())

# Common encodings:
print("Common encodings:")
for enc in ['utf-8', 'utf-8-sig', 'utf-16', 'latin-1', 'ascii', 'cp1252']:
    print(f"  {enc}")

English: Hello
Japanese: こんにちは
Arabic: مرحبا
Emoji: 🐍🎉

Common encodings:
  utf-8
  utf-8-sig
  utf-16
  latin-1
  ascii
  cp1252


In [7]:
# The 'errors' parameter controls what happens with bad characters
path = WORK / 'latin.txt'
path.write_bytes(b'caf\xe9')   # 'café' in latin-1

strategies = [
    ('strict',   'Raise UnicodeDecodeError  (default)'),
    ('ignore',   'Silently drop bad chars'),
    ('replace',  'Replace with ? or \ufffd'),
    ('backslashreplace', 'Replace with \\xNN escape'),
]

for strategy, desc in strategies:
    try:
        with open(path, 'r', encoding='utf-8', errors=strategy) as f:
            result = repr(f.read())
        print(f"  {strategy:20} → {result}  ({desc})")
    except UnicodeDecodeError as e:
        print(f"  {strategy:20} → UnicodeDecodeError: {e}")

  strict               → UnicodeDecodeError: 'utf-8' codec can't decode byte 0xe9 in position 3: unexpected end of data
  ignore               → 'caf'  (Silently drop bad chars)
  replace              → 'caf�'  (Replace with ? or �)
  backslashreplace     → 'caf\\xe9'  (Replace with \xNN escape)


---

## 3. Reading Files

In [8]:
# First, write a multi-line file to read from
poem_path = WORK / 'poem.txt'
poem_path.write_text(
    'Roses are red,\n'
    'Violets are blue,\n'
    'Python is awesome,\n'
    'And so are you!\n',
    encoding='utf-8'
)

# ── read() — read entire file into one string ─────────────────────────────
with open(poem_path, encoding='utf-8') as f:
    content = f.read()
print("read():")
print(repr(content))
print()

read():
'Roses are red,\nViolets are blue,\nPython is awesome,\nAnd so are you!\n'



In [9]:
# ── read(n) — read exactly n characters ──────────────────────────────────
with open(poem_path, encoding='utf-8') as f:
    chunk1 = f.read(6)
    chunk2 = f.read(6)
    print(f"First 6 chars:  {chunk1!r}")
    print(f"Next  6 chars:  {chunk2!r}")
    print(f"Position (tell): {f.tell()}")

    # seek() — move to a position in the file
    f.seek(0)       # back to the start
    print(f"After seek(0): {f.read(5)!r}")

First 6 chars:  'Roses '
Next  6 chars:  'are re'
Position (tell): 12
After seek(0): 'Roses'


In [10]:
# ── readline() — read one line at a time ─────────────────────────────────
with open(poem_path, encoding='utf-8') as f:
    line1 = f.readline()   # includes \n
    line2 = f.readline()
    print(f"Line 1: {line1!r}")
    print(f"Line 2: {line2!r}")

Line 1: 'Roses are red,\n'
Line 2: 'Violets are blue,\n'


In [11]:
# ── readlines() — read all lines into a list ─────────────────────────────
with open(poem_path, encoding='utf-8') as f:
    lines = f.readlines()   # list of strings, each with \n

print("readlines():")
print(lines)
print()

# Strip newlines when you don't need them
clean_lines = [line.rstrip('\n') for line in lines]
print("Stripped:", clean_lines)

readlines():
['Roses are red,\n', 'Violets are blue,\n', 'Python is awesome,\n', 'And so are you!\n']

Stripped: ['Roses are red,', 'Violets are blue,', 'Python is awesome,', 'And so are you!']


In [12]:
# ── Iterating directly — the most Pythonic and memory-efficient way ───────
# For large files this is much better than readlines() which loads all at once

print("Iterating line by line:")
with open(poem_path, encoding='utf-8') as f:
    for i, line in enumerate(f, start=1):
        print(f"  {i}: {line.rstrip()}")

Iterating line by line:
  1: Roses are red,
  2: Violets are blue,
  3: Python is awesome,
  4: And so are you!


---

## 4. Writing Files

In [13]:
# ── write() — write a string ──────────────────────────────────────────────
# ⚠️  'w' mode TRUNCATES an existing file! All previous content is lost.

path = WORK / 'log.txt'

with open(path, 'w', encoding='utf-8') as f:
    n = f.write('First line\n')   # returns number of characters written
    print(f"Wrote {n} characters")
    f.write('Second line\n')
    f.write('Third line\n')

print(path.read_text(encoding='utf-8'))

Wrote 11 characters
First line
Second line
Third line



In [14]:
# ── writelines() — write a list of strings (no automatic newlines!) ───────
lines = ['Alpha\n', 'Beta\n', 'Gamma\n']

with open(WORK / 'greek.txt', 'w', encoding='utf-8') as f:
    f.writelines(lines)   # writes each string as-is

print((WORK / 'greek.txt').read_text(encoding='utf-8'))

Alpha
Beta
Gamma



In [15]:
# ── Append mode — add to end without truncating ───────────────────────────
import datetime

log_path = WORK / 'app.log'

def log(message):
    ts = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    with open(log_path, 'a', encoding='utf-8') as f:
        f.write(f'[{ts}] {message}\n')

log('Server started')
log('User Alice logged in')
log('Request processed in 42ms')
log('Server stopped')

print(log_path.read_text(encoding='utf-8'))

[2026-05-17 12:39:33] Server started
[2026-05-17 12:39:33] User Alice logged in
[2026-05-17 12:39:33] Request processed in 42ms
[2026-05-17 12:39:33] Server stopped



In [16]:
# ── Exclusive creation — 'x' mode prevents overwriting ───────────────────
try:
    with open(WORK / 'log.txt', 'x') as f:   # log.txt already exists
        f.write('This will fail')
except FileExistsError as e:
    print(f"FileExistsError: {e}")

# Safe file creation pattern
new_path = WORK / 'brand_new_file.txt'
if not new_path.exists():
    with open(new_path, 'w', encoding='utf-8') as f:
        f.write('Created safely\n')
    print(f"Created: {new_path.name}")
else:
    print(f"File already exists: {new_path.name}")

FileExistsError: [Errno 17] File exists: 'file_io_demo/log.txt'
Created: brand_new_file.txt


In [17]:
# ── Flushing and buffering ────────────────────────────────────────────────
# Python buffers writes in memory for performance.
# Data may not hit disk until the buffer is flushed or the file is closed.

with open(WORK / 'buffered.txt', 'w', encoding='utf-8') as f:
    f.write('Line 1\n')
    f.flush()   # force write to OS immediately (useful for logs)
    f.write('Line 2\n')
    # implicit flush + close when with block exits

# Open with line buffering (flush after every newline)
with open(WORK / 'linebuf.txt', 'w', encoding='utf-8', buffering=1) as f:
    f.write('This flushes after the newline\n')  # immediate flush

print("Flushing demo complete.")

Flushing demo complete.


---

## 5. `pathlib` — Modern Path Handling

`pathlib` (Python 3.4+) provides an object-oriented alternative to string-based `os.path` operations. It's the recommended approach for new code.

In [18]:
from pathlib import Path

p = Path('file_io_demo/poem.txt')

# Path components
print(f"Path:     {p}")
print(f"Name:     {p.name}")        # poem.txt
print(f"Stem:     {p.stem}")        # poem
print(f"Suffix:   {p.suffix}")      # .txt
print(f"Suffixes: {p.suffixes}")    # ['.txt']
print(f"Parent:   {p.parent}")      # file_io_demo
print(f"Parts:    {p.parts}")
print(f"Absolute: {p.resolve()}")

Path:     file_io_demo/poem.txt
Name:     poem.txt
Stem:     poem
Suffix:   .txt
Suffixes: ['.txt']
Parent:   file_io_demo
Parts:    ('file_io_demo', 'poem.txt')
Absolute: /Users/jacob/Code/Notes/Tutorials/Python-Tutorial/Lessons/01-Beginner/06-FileIO/01-Basics/file_io_demo/poem.txt


In [19]:
# Building paths with / operator (cross-platform!)
home    = Path.home()
cwd     = Path.cwd()
subdir  = WORK / 'subdir' / 'nested'

print(f"Home: {home}")
print(f"CWD:  {cwd}")
print(f"Nested path: {subdir}")

# Create directory tree
subdir.mkdir(parents=True, exist_ok=True)
print(f"Created: {subdir.exists()}")

Home: /Users/jacob
CWD:  /Users/jacob/Code/Notes/Tutorials/Python-Tutorial/Lessons/01-Beginner/06-FileIO/01-Basics
Nested path: file_io_demo/subdir/nested
Created: True


In [20]:
# Testing paths
p = WORK / 'poem.txt'

print(f"exists():    {p.exists()}")
print(f"is_file():   {p.is_file()}")
print(f"is_dir():    {p.is_dir()}")
print(f"is_absolute:{p.is_absolute()}")

# Stat info
stat = p.stat()
print(f"Size:    {stat.st_size} bytes")
print(f"Modified:{datetime.datetime.fromtimestamp(stat.st_mtime):%Y-%m-%d %H:%M:%S}")

exists():    True
is_file():   True
is_dir():    False
is_absolute:False
Size:    68 bytes
Modified:2026-05-17 12:39:33


In [21]:
# Reading and writing with Path — the cleanest API
p = WORK / 'quick.txt'

# Write text
p.write_text('Hello from pathlib!\nLine two.\n', encoding='utf-8')

# Read text
content = p.read_text(encoding='utf-8')
print(repr(content))

# Write bytes
bp = WORK / 'quick.bin'
bp.write_bytes(b'\x00\x01\x02\xff')

# Read bytes
data = bp.read_bytes()
print(data)

'Hello from pathlib!\nLine two.\n'
b'\x00\x01\x02\xff'


In [22]:
# Globbing — finding files by pattern
print("All .txt files in WORK:")
for f in sorted(WORK.glob('*.txt')):
    print(f"  {f.name}  ({f.stat().st_size} bytes)")

print()
print("All files recursively:")
for f in sorted(WORK.rglob('*')):
    if f.is_file():
        rel = f.relative_to(WORK)
        print(f"  {rel}")

All .txt files in WORK:
  brand_new_file.txt  (15 bytes)
  buffered.txt  (14 bytes)
  dest.txt  (20 bytes)
  greek.txt  (17 bytes)
  hello.txt  (14 bytes)
  latin.txt  (4 bytes)
  linebuf.txt  (31 bytes)
  log.txt  (34 bytes)
  poem.txt  (68 bytes)
  quick.txt  (30 bytes)
  source.txt  (15 bytes)
  unicode_demo.txt  (76 bytes)

All files recursively:
  app.log
  brand_new_file.txt
  buffered.txt
  dest.txt
  greek.txt
  hello.txt
  latin.txt
  linebuf.txt
  log.txt
  poem.txt
  quick.bin
  quick.txt
  source.txt
  unicode_demo.txt


In [23]:
# Renaming, replacing, unlinking
src = WORK / 'brand_new_file.txt'
dst = WORK / 'renamed_file.txt'

if src.exists():
    src.rename(dst)   # rename within same filesystem
    print(f"Renamed to: {dst.name}")

# Replace — like rename but overwrites destination if it exists
# src.replace(dst)

# Delete a file
if dst.exists():
    dst.unlink()
    print(f"Deleted: {dst.name}")

# Delete a directory (must be empty)
empty_dir = WORK / 'empty'
empty_dir.mkdir(exist_ok=True)
empty_dir.rmdir()
print("Empty dir removed.")

# With Python 3.8+, missing_ok suppresses FileNotFoundError
(WORK / 'nonexistent.txt').unlink(missing_ok=True)
print("unlink(missing_ok=True) — no error even if file is absent.")

Renamed to: renamed_file.txt
Deleted: renamed_file.txt
Empty dir removed.
unlink(missing_ok=True) — no error even if file is absent.


---

## 6. Working with CSV Files

CSV (Comma-Separated Values) is the most common format for tabular data. Python's built-in `csv` module handles quoting, escaping, and different delimiters correctly — don't parse CSV manually with `split(',')`.

In [24]:
import csv

csv_path = WORK / 'employees.csv'

# Writing CSV
employees = [
    ['Name',    'Department',  'Salary', 'Start Date'],
    ['Alice',   'Engineering', 95000,   '2021-03-15'],
    ['Bob',     'Marketing',   72000,   '2019-07-01'],
    ['Carol',   'Engineering', 105000,  '2018-11-20'],
    ['Dave',    'HR',          68000,   '2022-01-10'],
    ['Eve',     'Engineering', 98000,   '2020-05-03'],
]

with open(csv_path, 'w', newline='', encoding='utf-8') as f:
    # newline='' is required on Windows to prevent extra blank lines
    writer = csv.writer(f)
    writer.writerows(employees)   # write all rows at once

print("CSV written:")
print(csv_path.read_text(encoding='utf-8'))

CSV written:
Name,Department,Salary,Start Date
Alice,Engineering,95000,2021-03-15
Bob,Marketing,72000,2019-07-01
Carol,Engineering,105000,2018-11-20
Dave,HR,68000,2022-01-10
Eve,Engineering,98000,2020-05-03



In [25]:
# Reading CSV
with open(csv_path, 'r', newline='', encoding='utf-8') as f:
    reader = csv.reader(f)
    header = next(reader)   # read the header row separately
    rows   = list(reader)

print(f"Header: {header}")
print(f"Rows ({len(rows)}):")
for row in rows:
    print(f"  {row}")

Header: ['Name', 'Department', 'Salary', 'Start Date']
Rows (5):
  ['Alice', 'Engineering', '95000', '2021-03-15']
  ['Bob', 'Marketing', '72000', '2019-07-01']
  ['Carol', 'Engineering', '105000', '2018-11-20']
  ['Dave', 'HR', '68000', '2022-01-10']
  ['Eve', 'Engineering', '98000', '2020-05-03']


In [26]:
# DictWriter / DictReader — work with dicts instead of lists
# Much more readable when you have many columns

csv_path2 = WORK / 'employees_dict.csv'
fieldnames = ['Name', 'Department', 'Salary', 'Start Date']

employees_data = [
    {'Name': 'Alice', 'Department': 'Engineering', 'Salary': 95000, 'Start Date': '2021-03-15'},
    {'Name': 'Bob',   'Department': 'Marketing',   'Salary': 72000, 'Start Date': '2019-07-01'},
    {'Name': 'Carol', 'Department': 'Engineering', 'Salary': 105000,'Start Date': '2018-11-20'},
]

with open(csv_path2, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()           # writes the header row
    writer.writerows(employees_data)

# Reading with DictReader — each row is an OrderedDict
with open(csv_path2, 'r', newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    print(f"Fields: {reader.fieldnames}")
    for row in reader:
        print(f"  {row['Name']:<8} | {row['Department']:<12} | ${int(row['Salary']):>7,}")

Fields: ['Name', 'Department', 'Salary', 'Start Date']
  Alice    | Engineering  | $ 95,000
  Bob      | Marketing    | $ 72,000
  Carol    | Engineering  | $105,000


In [27]:
# Dialects — handling non-standard CSV formats
tsv_path = WORK / 'data.tsv'

# Tab-separated values
with open(tsv_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f, delimiter='\t')
    writer.writerow(['City', 'Country', 'Population'])
    writer.writerow(['Tokyo',     'Japan',  13960000])
    writer.writerow(['São Paulo', 'Brazil', 12325000])
    writer.writerow(['New York',  'USA',    8336817])

with open(tsv_path, 'r', newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f, delimiter='\t')
    for row in reader:
        pop = int(row['Population'])
        print(f"  {row['City']:<12} ({row['Country']:<8}) — {pop:>12,}")

# Handle values that contain commas (csv does this automatically)
tricky_path = WORK / 'tricky.csv'
with open(tricky_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f, quoting=csv.QUOTE_ALL)
    writer.writerow(['Item',                 'Notes'])
    writer.writerow(['Widget, deluxe',       'Has a comma in the name'])
    writer.writerow(['Gadget "Pro"',         'Has quotes in the name'])
    writer.writerow(['Multi\nline\nitem',    'Has newlines'])

print("\nTricky CSV content:")
print(tricky_path.read_text(encoding='utf-8'))

  Tokyo        (Japan   ) —   13,960,000
  São Paulo    (Brazil  ) —   12,325,000
  New York     (USA     ) —    8,336,817

Tricky CSV content:
"Item","Notes"
"Widget, deluxe","Has a comma in the name"
"Gadget ""Pro""","Has quotes in the name"
"Multi
line
item","Has newlines"



---

## 7. Working with JSON Files

JSON is the universal format for configuration files, API responses, and data exchange. The `json` module handles all the Python ↔ JSON type conversions automatically.

In [28]:
import json

# Python types → JSON types
type_map = [
    ('dict',  {'a': 1},  'object  {}'),
    ('list',  [1, 2, 3], 'array   []'),
    ('tuple', (1, 2, 3), 'array   [] (tuples become arrays)'),
    ('str',   'hello',   'string  ""'),
    ('int',   42,        'number'),
    ('float', 3.14,      'number'),
    ('bool',  True,      'true / false'),
    ('None',  None,      'null'),
]
print(f"{'Python':8}  {'JSON output':18}  Notes")
print('-' * 55)
for pytype, value, note in type_map:
    print(f"{pytype:<8}  {json.dumps(value):<18}  {note}")

Python    JSON output         Notes
-------------------------------------------------------
dict      {"a": 1}            object  {}
list      [1, 2, 3]           array   []
tuple     [1, 2, 3]           array   [] (tuples become arrays)
str       "hello"             string  ""
int       42                  number
float     3.14                number
bool      true                true / false
None      null                null


In [29]:
# Writing JSON to a file
config = {
    "app_name": "MyApp",
    "version": "2.1.0",
    "debug": False,
    "database": {
        "host": "localhost",
        "port": 5432,
        "name": "mydb"
    },
    "allowed_hosts": ["localhost", "example.com"],
    "max_connections": 100,
    "timeout": None
}

json_path = WORK / 'config.json'

with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(config, f, indent=2, ensure_ascii=False)
    # indent=2 → pretty-print
    # ensure_ascii=False → allow non-ASCII characters (e.g. emoji, accents)

print(json_path.read_text(encoding='utf-8'))

{
  "app_name": "MyApp",
  "version": "2.1.0",
  "debug": false,
  "database": {
    "host": "localhost",
    "port": 5432,
    "name": "mydb"
  },
  "allowed_hosts": [
    "localhost",
    "example.com"
  ],
  "max_connections": 100,
  "timeout": null
}


In [30]:
# Reading JSON from a file
with open(json_path, 'r', encoding='utf-8') as f:
    loaded = json.load(f)

print(f"App: {loaded['app_name']} v{loaded['version']}")
print(f"DB host: {loaded['database']['host']}:{loaded['database']['port']}")
print(f"Hosts: {loaded['allowed_hosts']}")
print(f"Timeout is None: {loaded['timeout'] is None}")

App: MyApp v2.1.0
DB host: localhost:5432
Hosts: ['localhost', 'example.com']
Timeout is None: True


In [31]:
# Custom JSON serialisation — handling types json doesn't know about
import datetime, decimal, uuid

class CustomEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, datetime.datetime):
            return obj.isoformat()
        if isinstance(obj, datetime.date):
            return obj.isoformat()
        if isinstance(obj, decimal.Decimal):
            return str(obj)     # or float(obj) if precision isn't critical
        if isinstance(obj, uuid.UUID):
            return str(obj)
        if isinstance(obj, set):
            return sorted(obj)  # sets → sorted lists
        return super().default(obj)   # raises TypeError for unknown types

data = {
    'created_at': datetime.datetime.now(),
    'birth_date': datetime.date(1990, 6, 15),
    'price':      decimal.Decimal('19.99'),
    'user_id':    uuid.uuid4(),
    'tags':       {'python', 'tutorial', 'io'},
}

json_str = json.dumps(data, cls=CustomEncoder, indent=2)
print(json_str)

{
  "created_at": "2026-05-17T12:39:33.593882",
  "birth_date": "1990-06-15",
  "price": "19.99",
  "user_id": "cb4afbee-fd59-45d9-a465-585528616ff9",
  "tags": [
    "io",
    "python",
    "tutorial"
  ]
}


In [32]:
# JSON Lines format — one JSON object per line, great for streaming/logs
jsonl_path = WORK / 'events.jsonl'

events = [
    {'ts': '2026-05-17T09:00:00', 'event': 'login',  'user': 'alice'},
    {'ts': '2026-05-17T09:01:23', 'event': 'search', 'query': 'python io'},
    {'ts': '2026-05-17T09:02:45', 'event': 'click',  'url': '/docs/io'},
    {'ts': '2026-05-17T09:10:00', 'event': 'logout', 'user': 'alice'},
]

# Write JSONL
with open(jsonl_path, 'w', encoding='utf-8') as f:
    for event in events:
        f.write(json.dumps(event) + '\n')   # one compact JSON per line

# Read JSONL — one object at a time (memory-efficient for large files)
print("Events from JSONL:")
with open(jsonl_path, 'r', encoding='utf-8') as f:
    for line in f:
        event = json.loads(line)
        print(f"  [{event['ts']}] {event['event']}")

Events from JSONL:
  [2026-05-17T09:00:00] login
  [2026-05-17T09:01:23] search
  [2026-05-17T09:02:45] click
  [2026-05-17T09:10:00] logout


---

## 8. Binary Files

Binary mode (`'rb'` / `'wb'`) reads and writes raw bytes. Essential for images, audio, executables, and any non-text format.

In [33]:
# Writing and reading binary data
import struct

bin_path = WORK / 'numbers.bin'

# Pack Python values into binary using struct
# Format: 3 unsigned shorts (H) + 1 float (f)
fmt = '>3Hf'   # > = big-endian, H = unsigned short, f = float

with open(bin_path, 'wb') as f:
    for r, g, b, alpha in [(255, 0, 0, 1.0), (0, 255, 0, 0.5), (0, 0, 255, 0.25)]:
        f.write(struct.pack(fmt, r, g, b, alpha))

print(f"File size: {bin_path.stat().st_size} bytes")

# Read it back
record_size = struct.calcsize(fmt)   # bytes per record
print(f"Record size: {record_size} bytes")

with open(bin_path, 'rb') as f:
    while chunk := f.read(record_size):
        if len(chunk) == record_size:
            r, g, b, alpha = struct.unpack(fmt, chunk)
            print(f"  RGB({r:3}, {g:3}, {b:3}) alpha={alpha:.2f}")

File size: 30 bytes
Record size: 10 bytes
  RGB(255,   0,   0) alpha=1.00
  RGB(  0, 255,   0) alpha=0.50
  RGB(  0,   0, 255) alpha=0.25


In [34]:
# Copying a binary file in chunks — memory-efficient for large files
def copy_file(src: Path, dst: Path, chunk_size: int = 65536) -> int:
    """Copy a file in chunks. Returns bytes copied."""
    total = 0
    with open(src, 'rb') as fin, open(dst, 'wb') as fout:
        while chunk := fin.read(chunk_size):
            fout.write(chunk)
            total += len(chunk)
    return total

copied = copy_file(bin_path, WORK / 'numbers_copy.bin')
print(f"Copied {copied} bytes")

Copied 30 bytes


In [35]:
# BytesIO — an in-memory binary file (no disk I/O)
import io

buf = io.BytesIO()
buf.write(b'\x89PNG\r\n\x1a\n')   # pretend we're building a file
buf.write(b'\x00' * 16)
buf.write(b'IEND')

# Read from the start
buf.seek(0)
data = buf.read()
print(f"In-memory bytes: {data[:8]}...  total {len(data)} bytes")

# Useful for processing binary data without touching disk
# (e.g. generating a ZIP file in memory and returning it from a web API)

# StringIO — the text equivalent
sbuf = io.StringIO()
sbuf.write('Line 1\n')
sbuf.write('Line 2\n')
sbuf.seek(0)
print(sbuf.read())

In-memory bytes: b'\x89PNG\r\n\x1a\n'...  total 28 bytes
Line 1
Line 2



In [36]:
# Working with ZIP files
import zipfile

zip_path = WORK / 'archive.zip'

# Create a ZIP
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(WORK / 'poem.txt',      arcname='poem.txt')  # add a file
    zf.write(WORK / 'config.json',   arcname='config.json')
    zf.writestr('readme.txt', 'Auto-generated archive\n')  # add from string

print(f"ZIP size: {zip_path.stat().st_size} bytes")

# Inspect the ZIP
with zipfile.ZipFile(zip_path, 'r') as zf:
    print("Contents:")
    for info in zf.infolist():
        print(f"  {info.filename:<20} {info.file_size:>6} bytes")

    # Extract a specific file
    content = zf.read('readme.txt')
    print(f"readme.txt: {content!r}")

    # Extract all files
    zf.extractall(WORK / 'extracted')

ZIP size: 555 bytes
Contents:
  poem.txt                 68 bytes
  config.json             254 bytes
  readme.txt               23 bytes
readme.txt: b'Auto-generated archive\n'


---

## 9. Temporary Files & Directories

The `tempfile` module creates files and directories that are automatically cleaned up.

In [37]:
import tempfile

# NamedTemporaryFile — auto-deleted when closed (delete=True by default)
with tempfile.NamedTemporaryFile(mode='w', suffix='.txt',
                                  encoding='utf-8', delete=True) as f:
    print(f"Temp file: {f.name}")
    f.write('Temporary data\n')
    f.flush()
    print(f"  Size: {Path(f.name).stat().st_size} bytes")
# File is deleted here

print(f"File gone: {not Path(f.name).exists()}")

Temp file: /var/folders/t4/gbnsg7zd4730vwvtrm2b19740000gn/T/tmp6y9q5k8m.txt
  Size: 15 bytes
File gone: True


In [38]:
# TemporaryDirectory — entire directory tree cleaned up
with tempfile.TemporaryDirectory() as tmpdir:
    tmpdir = Path(tmpdir)
    print(f"Temp dir: {tmpdir}")

    # Use it like any other directory
    (tmpdir / 'data.txt').write_text('temporary data')
    (tmpdir / 'cache').mkdir()
    (tmpdir / 'cache' / 'item.bin').write_bytes(b'\xff' * 10)

    files = list(tmpdir.rglob('*'))
    print(f"  Created {len(files)} items")

print(f"Directory gone: {not tmpdir.exists()}")

Temp dir: /var/folders/t4/gbnsg7zd4730vwvtrm2b19740000gn/T/tmpui6yi5k5
  Created 3 items
Directory gone: True


In [39]:
# mkstemp / mkdtemp — manual control (YOU must delete them)
fd, path = tempfile.mkstemp(suffix='.tmp', prefix='myapp_')
path = Path(path)
print(f"mkstemp: {path.name}")

try:
    os.write(fd, b'data written via file descriptor\n')
    os.close(fd)
    print(f"  Content: {path.read_bytes()}")
finally:
    path.unlink()   # you must clean up manually!

# Atomic file write pattern — write to temp, then rename (atomic on POSIX)
def atomic_write(path: Path, content: str, encoding='utf-8'):
    """Write content to a file atomically — no partial writes visible."""
    tmp_fd, tmp_path = tempfile.mkstemp(dir=path.parent, prefix='.tmp_')
    tmp_path = Path(tmp_path)
    try:
        os.write(tmp_fd, content.encode(encoding))
        os.close(tmp_fd)
        tmp_path.replace(path)   # atomic rename
    except Exception:
        os.close(tmp_fd)
        tmp_path.unlink(missing_ok=True)
        raise

atomic_write(WORK / 'atomic.txt', 'Written atomically!\n')
print((WORK / 'atomic.txt').read_text())

mkstemp: myapp_i1x_02dy.tmp
  Content: b'data written via file descriptor\n'
Written atomically!



---

## 10. `os` & `shutil` — File System Operations

In [40]:
import os, shutil

# ── os — lower-level OS interface ────────────────────────────────────────

# Directory operations
os.makedirs(str(WORK / 'nested/deep'), exist_ok=True)
print("os.makedirs done")

# List directory contents
entries = os.listdir(str(WORK))
print(f"os.listdir: {sorted(entries)[:5]}...")

# scandir — like listdir but returns DirEntry objects with stat info
print("\nos.scandir (files only):")
with os.scandir(str(WORK)) as it:
    for entry in it:
        if entry.is_file():
            print(f"  {entry.name:25} {entry.stat().st_size:>6} bytes")

os.makedirs done
os.listdir: ['app.log', 'archive.zip', 'atomic.txt', 'buffered.txt', 'config.json']...

os.scandir (files only):
  unicode_demo.txt              76 bytes
  numbers.bin                   30 bytes
  atomic.txt                    20 bytes
  numbers_copy.bin              30 bytes
  config.json                  254 bytes
  poem.txt                      68 bytes
  employees_dict.csv           140 bytes
  source.txt                    15 bytes
  data.tsv                      97 bytes
  log.txt                       34 bytes
  events.jsonl                 269 bytes
  latin.txt                      4 bytes
  dest.txt                      20 bytes
  linebuf.txt                   31 bytes
  buffered.txt                  14 bytes
  tricky.csv                   137 bytes
  employees.csv                200 bytes
  archive.zip                  555 bytes
  quick.bin                      4 bytes
  quick.txt                     30 bytes
  app.log                      165 bytes
  greek.t

In [41]:
# os.walk — traverse a directory tree
print("Directory tree:")
for root, dirs, files in os.walk(str(WORK)):
    depth = root.replace(str(WORK), '').count(os.sep)
    indent = '  ' * depth
    print(f"{indent}{Path(root).name}/")
    for fname in sorted(files):
        fpath = Path(root) / fname
        print(f"{indent}  {fname} ({fpath.stat().st_size}b)")

    # Prune the walk — skip directories named 'extracted' or 'subdir'
    dirs[:] = [d for d in dirs if d not in ('extracted', 'nested')]

Directory tree:
file_io_demo/
  app.log (165b)
  archive.zip (555b)
  atomic.txt (20b)
  buffered.txt (14b)
  config.json (254b)
  data.tsv (97b)
  dest.txt (20b)
  employees.csv (200b)
  employees_dict.csv (140b)
  events.jsonl (269b)
  greek.txt (17b)
  hello.txt (14b)
  latin.txt (4b)
  linebuf.txt (31b)
  log.txt (34b)
  numbers.bin (30b)
  numbers_copy.bin (30b)
  poem.txt (68b)
  quick.bin (4b)
  quick.txt (30b)
  source.txt (15b)
  tricky.csv (137b)
  unicode_demo.txt (76b)
  subdir/


In [42]:
# ── shutil — high-level file operations ──────────────────────────────────

# Copy a file (content only)
shutil.copy(str(WORK / 'poem.txt'), str(WORK / 'poem_copy.txt'))
print("shutil.copy done")

# Copy with metadata (permissions, timestamps)
shutil.copy2(str(WORK / 'poem.txt'), str(WORK / 'poem_copy2.txt'))
print("shutil.copy2 done")

# Copy an entire directory tree
if (WORK / 'backup').exists():
    shutil.rmtree(str(WORK / 'backup'))
shutil.copytree(str(WORK / 'extracted'), str(WORK / 'backup'))
print(f"Backed up: {list((WORK / 'backup').iterdir())}")

# Move a file or directory
shutil.move(str(WORK / 'poem_copy.txt'),  str(WORK / 'moved_poem.txt'))
print("shutil.move done")

# Delete an entire directory tree
shutil.rmtree(str(WORK / 'backup'))
print(f"rmtree done — backup exists: {(WORK / 'backup').exists()}")

# Disk usage
usage = shutil.disk_usage('/')
print(f"\nDisk usage:")
print(f"  Total: {usage.total / 1e9:.1f} GB")
print(f"  Used:  {usage.used  / 1e9:.1f} GB")
print(f"  Free:  {usage.free  / 1e9:.1f} GB")

shutil.copy done
shutil.copy2 done
Backed up: [PosixPath('file_io_demo/backup/config.json'), PosixPath('file_io_demo/backup/poem.txt'), PosixPath('file_io_demo/backup/readme.txt')]
shutil.move done
rmtree done — backup exists: False

Disk usage:
  Total: 994.7 GB
  Used:  967.0 GB
  Free:  27.7 GB


---

## 11. File Metadata & `stat`

In [43]:
import os, stat, datetime

p = WORK / 'config.json'
s = p.stat()

print(f"File:          {p.name}")
print(f"Size:          {s.st_size:,} bytes")
print(f"Created:       {datetime.datetime.fromtimestamp(s.st_ctime):%Y-%m-%d %H:%M:%S}")
print(f"Modified:      {datetime.datetime.fromtimestamp(s.st_mtime):%Y-%m-%d %H:%M:%S}")
print(f"Accessed:      {datetime.datetime.fromtimestamp(s.st_atime):%Y-%m-%d %H:%M:%S}")
print(f"Inode:         {s.st_ino}")
print(f"Hard links:    {s.st_nlink}")
print(f"Mode (octal):  {oct(s.st_mode)}")

# Check permissions
readable   = os.access(p, os.R_OK)
writable   = os.access(p, os.W_OK)
executable = os.access(p, os.X_OK)
print(f"Readable:  {readable}")
print(f"Writable:  {writable}")
print(f"Executable:{executable}")

File:          config.json
Size:          254 bytes
Created:       2026-05-17 12:39:33
Modified:      2026-05-17 12:39:33
Accessed:      2026-05-17 12:39:33
Inode:         349998092
Hard links:    1
Mode (octal):  0o100644
Readable:  True
Writable:  True
Executable:False


In [44]:
# Touch a file (create or update its timestamp)
touch_path = WORK / 'touched.txt'
touch_path.touch()   # creates if absent, updates mtime if present

# Set a specific modification time
import time
target_time = time.mktime(datetime.datetime(2025, 1, 1).timetuple())
os.utime(touch_path, (target_time, target_time))   # (atime, mtime)

mtime = datetime.datetime.fromtimestamp(touch_path.stat().st_mtime)
print(f"Set mtime to: {mtime:%Y-%m-%d}")

# Find the largest files in a directory
print("\nLargest files in WORK:")
files = [(f, f.stat().st_size) for f in WORK.rglob('*') if f.is_file()]
for f, size in sorted(files, key=lambda x: x[1], reverse=True)[:5]:
    print(f"  {size:>8,} bytes  {f.relative_to(WORK)}")

Set mtime to: 2025-01-01

Largest files in WORK:
       555 bytes  archive.zip
       269 bytes  events.jsonl
       254 bytes  config.json
       254 bytes  extracted/config.json
       200 bytes  employees.csv


---

## 12. Large File Strategies — Streaming & Memory-Mapping

In [45]:
# Generate a large-ish CSV for testing
import random, csv

large_csv = WORK / 'large.csv'
random.seed(0)

with open(large_csv, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['id', 'value', 'category'])
    for i in range(50_000):
        writer.writerow([i, round(random.gauss(100, 15), 2),
                         random.choice(['A', 'B', 'C'])])

size_mb = large_csv.stat().st_size / 1e6
print(f"Generated {large_csv.name}: {size_mb:.2f} MB, 50,000 rows")

Generated large.csv: 0.76 MB, 50,000 rows


In [46]:
# ── Strategy 1: Process line-by-line (O(1) memory) ────────────────────────
import time

start = time.perf_counter()
total, count = 0.0, 0

with open(large_csv, 'r', newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f)
    for row in reader:          # each row loaded one at a time
        total += float(row['value'])
        count += 1

elapsed = time.perf_counter() - start
print(f"Streaming: avg={total/count:.2f}, rows={count:,}, time={elapsed*1000:.0f}ms")

Streaming: avg=100.05, rows=50,000, time=44ms


In [47]:
# ── Strategy 2: Chunked reading ───────────────────────────────────────────
def read_in_chunks(path, chunk_size=8192):
    """Generator that yields file content in chunks of bytes."""
    with open(path, 'rb') as f:
        while chunk := f.read(chunk_size):
            yield chunk

# Count occurrences of 'A' category across the whole file in chunks
count_a = 0
for chunk in read_in_chunks(large_csv):
    count_a += chunk.count(b',A\r') + chunk.count(b',A\n')

print(f"Category A rows (approx): {count_a:,}")

Category A rows (approx): 16,775


In [48]:
# ── Strategy 3: Memory-mapping — random access without loading the file ────
import mmap

with open(large_csv, 'rb') as f:
    with mmap.mmap(f.fileno(), 0, access=mmap.ACCESS_READ) as mm:
        # The file is not in RAM — the OS maps it on demand
        print(f"mmap size: {len(mm):,} bytes")

        # Random access — jump to a position instantly
        mm.seek(0)
        first_line = mm.readline()
        print(f"Header: {first_line}")

        # Search for a byte pattern
        idx = mm.find(b',C,')
        print(f"First ',C,' at byte offset: {idx:,}")

        # Count all occurrences
        content = bytes(mm)
        cat_b_count = content.count(b',B,')
        print(f"',B,' occurrences: {cat_b_count:,}")

mmap size: 758,881 bytes
Header: b'id,value,category\r\n'
First ',C,' at byte offset: -1
',B,' occurrences: 0


---

## 13. Error Handling in File I/O

In [49]:
# Common file exceptions and when they occur
exceptions = [
    ('FileNotFoundError',  'File or directory does not exist'),
    ('FileExistsError',    "File already exists (mode='x')"),
    ('PermissionError',    'No read/write permission'),
    ('IsADirectoryError',  'Tried to open a directory as a file'),
    ('NotADirectoryError', 'Path component is a file, not a directory'),
    ('UnicodeDecodeError', 'File bytes are invalid for the encoding'),
    ('UnicodeEncodeError', 'String cannot be encoded to bytes'),
    ('OSError',            'General OS-level I/O error (parent of all above)'),
]

print(f"{'Exception':<25} Cause")
print('-' * 60)
for exc, cause in exceptions:
    print(f"  {exc:<23} {cause}")

Exception                 Cause
------------------------------------------------------------
  FileNotFoundError       File or directory does not exist
  FileExistsError         File already exists (mode='x')
  PermissionError         No read/write permission
  IsADirectoryError       Tried to open a directory as a file
  NotADirectoryError      Path component is a file, not a directory
  UnicodeDecodeError      File bytes are invalid for the encoding
  UnicodeEncodeError      String cannot be encoded to bytes
  OSError                 General OS-level I/O error (parent of all above)


In [50]:
# Triggering and catching each exception type
test_cases = [
    (lambda: open('/nonexistent/path.txt'), FileNotFoundError),
    (lambda: open(WORK / 'config.json', 'x'), FileExistsError),
    (lambda: open('/root/secret.txt'), PermissionError),
    (lambda: open('/etc'), IsADirectoryError),
]

for fn, expected_exc in test_cases:
    try:
        f = fn()
        f.close()
        print(f"  {expected_exc.__name__:25}: not raised (file was accessible)")
    except expected_exc as e:
        print(f"  {expected_exc.__name__:25}: caught — {e}")
    except OSError as e:
        print(f"  OSError (instead of {expected_exc.__name__:15}): {e}")

  FileNotFoundError        : caught — [Errno 2] No such file or directory: '/nonexistent/path.txt'
  FileExistsError          : caught — [Errno 17] File exists: 'file_io_demo/config.json'
  OSError (instead of PermissionError): [Errno 2] No such file or directory: '/root/secret.txt'
  IsADirectoryError        : caught — [Errno 21] Is a directory: '/etc'


In [51]:
# Robust file reading with full error handling
def safe_read(path, encoding='utf-8', default=None):
    """
    Read a file safely, returning default if anything goes wrong.
    Logs the specific error for debugging.
    """
    try:
        with open(path, 'r', encoding=encoding) as f:
            return f.read()
    except FileNotFoundError:
        print(f"[WARN] File not found: {path}")
    except PermissionError:
        print(f"[ERROR] Permission denied: {path}")
    except UnicodeDecodeError as e:
        print(f"[ERROR] Encoding error in {path}: {e}")
    except OSError as e:
        print(f"[ERROR] OS error reading {path}: {e}")
    return default

# Test it
content = safe_read(WORK / 'poem.txt')
print(f"Read {len(content)} chars")

missing = safe_read(WORK / 'no_such_file.txt', default='')
print(f"Missing file returned: {missing!r}")

Read 68 chars
[WARN] File not found: file_io_demo/no_such_file.txt
Missing file returned: ''


In [52]:
# Atomic write with rollback on error
def safe_write(path: Path, content: str, encoding='utf-8'):
    """
    Write content atomically:
    - Write to a temp file first
    - Rename to final path only on success
    - If writing fails, original file is untouched
    """
    import tempfile
    tmp_fd, tmp_path = tempfile.mkstemp(
        dir=path.parent, prefix='.tmp_', suffix=path.suffix
    )
    tmp_path = Path(tmp_path)
    try:
        with os.fdopen(tmp_fd, 'w', encoding=encoding) as f:
            f.write(content)
        tmp_path.replace(path)    # atomic on POSIX, near-atomic on Windows
        return True
    except Exception as e:
        print(f"Write failed: {e}")
        tmp_path.unlink(missing_ok=True)
        return False

ok = safe_write(WORK / 'safe_output.txt', 'Safely written!\n')
print(f"Write succeeded: {ok}")
print((WORK / 'safe_output.txt').read_text())

Write succeeded: True
Safely written!



---

## 14. `configparser` — INI Config Files

Many applications use `.ini` config files. Python's `configparser` reads and writes them easily.

In [53]:
import configparser

config = configparser.ConfigParser()

# Build config programmatically
config['DEFAULT'] = {
    'debug': 'false',
    'log_level': 'INFO',
}

config['database'] = {
    'host': 'localhost',
    'port': '5432',
    'name': 'mydb',
    'pool_size': '10',
}

config['server'] = {
    'host': '0.0.0.0',
    'port': '8080',
    'workers': '4',
    'debug': 'true',   # overrides DEFAULT
}

config['email'] = {
    'smtp_host': 'smtp.example.com',
    'smtp_port': '587',
    'use_tls': 'true',
}

ini_path = WORK / 'app.ini'
with open(ini_path, 'w') as f:
    config.write(f)

print(ini_path.read_text())

[DEFAULT]
debug = false
log_level = INFO

[database]
host = localhost
port = 5432
name = mydb
pool_size = 10

[server]
host = 0.0.0.0
port = 8080
workers = 4
debug = true

[email]
smtp_host = smtp.example.com
smtp_port = 587
use_tls = true




In [54]:
# Reading an INI file
cfg = configparser.ConfigParser()
cfg.read(ini_path)

print(f"Sections: {cfg.sections()}")
print()

# Access values
print(f"DB host:     {cfg['database']['host']}")
print(f"DB port:     {cfg.getint('database', 'port')}")
print(f"Pool size:   {cfg.getint('database', 'pool_size')}")
print(f"Server debug:{cfg.getboolean('server', 'debug')}")
print(f"TLS:         {cfg.getboolean('email', 'use_tls')}")

# DEFAULT values are inherited by all sections
print(f"\nDB log_level (from DEFAULT): {cfg['database']['log_level']}")

# Safe access with fallback
timeout = cfg.getint('database', 'timeout', fallback=30)
print(f"DB timeout (fallback):       {timeout}s")

Sections: ['database', 'server', 'email']

DB host:     localhost
DB port:     5432
Pool size:   10
Server debug:True
TLS:         True

DB log_level (from DEFAULT): INFO
DB timeout (fallback):       30s


---

## 15. `pickle` — Python Object Serialisation

`pickle` converts any Python object (lists, dicts, custom classes, functions) to bytes and back. Useful for caching and saving program state.

In [55]:
import pickle

# Save any Python object to a file
data = {
    'model_name': 'LinearRegression',
    'coefficients': [2.5, -1.3, 0.8, 4.1],
    'intercept': 0.42,
    'feature_names': ['age', 'income', 'score', 'visits'],
    'trained_at': datetime.datetime.now(),
}

pkl_path = WORK / 'model.pkl'

# Serialize (pickle)
with open(pkl_path, 'wb') as f:
    pickle.dump(data, f, protocol=pickle.HIGHEST_PROTOCOL)

print(f"Saved {pkl_path.stat().st_size} bytes")

# Deserialize (unpickle)
with open(pkl_path, 'rb') as f:
    loaded = pickle.load(f)

print(f"Model: {loaded['model_name']}")
print(f"Coefficients: {loaded['coefficients']}")
print(f"Trained at: {loaded['trained_at']:%Y-%m-%d %H:%M}")

Saved 230 bytes
Model: LinearRegression
Coefficients: [2.5, -1.3, 0.8, 4.1]
Trained at: 2026-05-17 12:39


In [56]:
# Pickling custom objects
from dataclasses import dataclass

@dataclass
class Experiment:
    name: str
    params: dict
    results: list
    notes: str = ''

exp = Experiment(
    name='exp_001',
    params={'lr': 0.01, 'epochs': 100, 'batch_size': 32},
    results=[0.85, 0.87, 0.91, 0.93],
    notes='Initial baseline run'
)

# Pickle and restore
pkl_bytes = pickle.dumps(exp)          # to bytes (no file)
restored  = pickle.loads(pkl_bytes)    # from bytes

print(f"Restored: {restored}")
print(f"Same data: {exp == restored}")

# ⚠️ SECURITY WARNING:
# Never unpickle data from an untrusted source!
# Pickle can execute arbitrary code during deserialization.
# Use JSON for data exchange with external systems.
print()
print("⚠️  Only unpickle data you created yourself or trust completely.")

Restored: Experiment(name='exp_001', params={'lr': 0.01, 'epochs': 100, 'batch_size': 32}, results=[0.85, 0.87, 0.91, 0.93], notes='Initial baseline run')
Same data: True

⚠️  Only unpickle data you created yourself or trust completely.


In [57]:
# shelve — persistent dict backed by pickle
import shelve

shelf_path = str(WORK / 'cache')   # shelve adds its own extensions

# Write
with shelve.open(shelf_path) as db:
    db['user:1'] = {'name': 'Alice', 'score': 95}
    db['user:2'] = {'name': 'Bob',   'score': 82}
    db['config'] = {'version': '1.0', 'debug': False}

# Read — works like a dict, persists across runs
with shelve.open(shelf_path) as db:
    print(f"Keys: {list(db.keys())}")
    print(f"User 1: {db['user:1']}")
    print(f"User 2: {db['user:2']}")

Keys: ['config', 'user:1', 'user:2']
User 1: {'name': 'Alice', 'score': 95}
User 2: {'name': 'Bob', 'score': 82}


---

## 16. Quick Reference

| Task | Tool / Syntax |
|---|---|
| Open a file safely | `with open(path, mode, encoding='utf-8') as f:` |
| Read entire file | `f.read()` or `Path(p).read_text()` |
| Read line by line | `for line in f:` |
| Read N bytes | `f.read(n)` |
| Write text | `f.write(s)` or `Path(p).write_text(s)` |
| Append to file | `open(p, 'a')` |
| Seek / tell | `f.seek(pos)` / `f.tell()` |
| Force write | `f.flush()` |
| Build paths | `Path('a') / 'b' / 'c.txt'` |
| Check existence | `p.exists()`, `p.is_file()`, `p.is_dir()` |
| Find files | `p.glob('*.csv')`, `p.rglob('**/*.py')` |
| File metadata | `p.stat()` — size, mtime, ctime |
| Read CSV | `csv.DictReader(f)` |
| Write CSV | `csv.DictWriter(f, fieldnames)` |
| Read JSON | `json.load(f)` / `json.loads(s)` |
| Write JSON | `json.dump(obj, f, indent=2)` |
| Copy file | `shutil.copy2(src, dst)` |
| Copy tree | `shutil.copytree(src, dst)` |
| Delete tree | `shutil.rmtree(path)` |
| Temp file | `tempfile.NamedTemporaryFile()` |
| Temp dir | `tempfile.TemporaryDirectory()` |
| In-memory file | `io.StringIO()` / `io.BytesIO()` |
| ZIP archive | `zipfile.ZipFile(p, 'w')` |
| Config file | `configparser.ConfigParser()` |
| Pickle object | `pickle.dump(obj, f)` / `pickle.load(f)` |
| Memory-map | `mmap.mmap(f.fileno(), 0)` |

In [58]:
# 🏆 Practice: Build a simple file-based database
import json, csv, datetime
from pathlib import Path

class SimpleDB:
    """
    A tiny file-backed database using JSON for storage.
    Demonstrates open, read, write, and error handling together.
    """

    def __init__(self, path: Path):
        self.path = Path(path)
        self._data = self._load()

    def _load(self) -> dict:
        if not self.path.exists():
            return {}
        try:
            return json.loads(self.path.read_text(encoding='utf-8'))
        except (json.JSONDecodeError, OSError) as e:
            print(f"[WARN] Could not load DB: {e} — starting empty.")
            return {}

    def _save(self):
        # Atomic write — temp file then replace
        import tempfile, os
        tmp_fd, tmp = tempfile.mkstemp(dir=self.path.parent)
        try:
            with os.fdopen(tmp_fd, 'w', encoding='utf-8') as f:
                json.dump(self._data, f, indent=2, default=str)
            Path(tmp).replace(self.path)
        except Exception:
            Path(tmp).unlink(missing_ok=True)
            raise

    def set(self, key: str, value) -> None:
        self._data[key] = value
        self._save()

    def get(self, key: str, default=None):
        return self._data.get(key, default)

    def delete(self, key: str) -> bool:
        if key in self._data:
            del self._data[key]
            self._save()
            return True
        return False

    def keys(self):
        return list(self._data.keys())

    def export_csv(self, path: Path):
        """Export all key-value pairs to CSV."""
        with open(path, 'w', newline='', encoding='utf-8') as f:
            writer = csv.writer(f)
            writer.writerow(['key', 'value'])
            for k, v in self._data.items():
                writer.writerow([k, json.dumps(v)])
        print(f"Exported {len(self._data)} records to {path.name}")

    def __len__(self):
        return len(self._data)

    def __repr__(self):
        return f"SimpleDB({self.path.name!r}, {len(self)} records)"


# --- Demo ---
db = SimpleDB(WORK / 'mydb.json')
print(db)

db.set('user:1', {'name': 'Alice', 'score': 95, 'joined': str(datetime.date.today())})
db.set('user:2', {'name': 'Bob',   'score': 82, 'joined': '2024-01-15'})
db.set('config', {'version': '1.0', 'max_users': 100})

print(f"Keys: {db.keys()}")
print(f"User 1: {db.get('user:1')}")
print(f"Missing key: {db.get('nonexistent', 'DEFAULT')}")

db.delete('config')
print(f"After delete: {db.keys()}")

db.export_csv(WORK / 'db_export.csv')
print()
print("DB JSON file:")
print((WORK / 'mydb.json').read_text())

SimpleDB('mydb.json', 0 records)
Keys: ['user:1', 'user:2', 'config']
User 1: {'name': 'Alice', 'score': 95, 'joined': '2026-05-17'}
Missing key: DEFAULT
After delete: ['user:1', 'user:2']
Exported 2 records to db_export.csv

DB JSON file:
{
  "user:1": {
    "name": "Alice",
    "score": 95,
    "joined": "2026-05-17"
  },
  "user:2": {
    "name": "Bob",
    "score": 82,
    "joined": "2024-01-15"
  }
}


In [59]:
# Cleanup — remove working directory
import shutil
# shutil.rmtree(WORK)   # uncomment to delete all demo files
print(f"Demo files in: {WORK.resolve()}")
print(f"Files created: {len(list(WORK.rglob('*')))}")

Demo files in: /Users/jacob/Code/Notes/Tutorials/Python-Tutorial/Lessons/01-Beginner/06-FileIO/01-Basics/file_io_demo
Files created: 41
